In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
# Standard Library imports
from pathlib import Path
from datetime import datetime
from tqdm import trange

# External imports
import cv2
import supervision as sv

In [ ]:
# Monkey-patch supervision's coco_annotations_to_masks to correctly handle
# multi-polygon segmentations.
#
# Problem: In COCO format, a single annotation's "segmentation" field can be a
# list of *multiple* polygons (e.g. a fish whose body is split into separate
# visible blobs by occlusion). The upstream implementation flattens all polygons
# into one coordinate array and rasterizes it as a single polygon, producing a
# corrupt mask that connects unrelated blobs with spurious edges.
#
# Fix: Rasterize each polygon independently (via supervision's own
# _polygons_to_masks from the YOLO format module), then merge them into one
# binary mask with np.logical_or.reduce. This preserves disjoint blobs as
# separate regions within the same detection mask.
#
# The patch is applied by overwriting the module-level reference so that
# coco_annotations_to_detections (which calls coco_annotations_to_masks by
# name) picks up the fixed version at call time.

import numpy as np
import supervision.dataset.formats.coco as _coco_fmt
from supervision.dataset.formats.yolo import _polygons_to_masks
from supervision.dataset.utils import rle_to_mask


def _patched_coco_annotations_to_masks(image_annotations, resolution_wh):
    masks = []
    for ann in image_annotations:
        if ann["iscrowd"]:
            mask = rle_to_mask(
                rle=np.array(ann["segmentation"]["counts"]),
                resolution_wh=resolution_wh,
            )
        else:
            polygons = [
                np.reshape(np.asarray(poly, dtype=np.int32), (-1, 2))
                for poly in ann["segmentation"]
            ]
            per_polygon_masks = _polygons_to_masks(
                polygons=polygons, resolution_wh=resolution_wh
            )
            mask = np.logical_or.reduce(per_polygon_masks)
        masks.append(mask)
    return np.array(masks, dtype=bool)


_coco_fmt.coco_annotations_to_masks = _patched_coco_annotations_to_masks

In [ ]:
ROOT_PATH = Path(<FILL_ME>)

In [ ]:
coco_dirpath = str(ROOT_PATH / "dataset_coco")

ds_train = sv.DetectionDataset.from_coco(
    images_directory_path=f"{coco_dirpath}/images/train",
    annotations_path=f"{coco_dirpath}/annotations/instances_train.json",
    force_masks=True
)

In [ ]:
ds_train.image_paths = sorted(ds_train.image_paths)

In [ ]:
ds_train.classes

In [ ]:
len(ds_train)

### Visualization of frames

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
mask_annotator = sv.MaskAnnotator()

start_image_idx = 0
num_images = 1
end_image_idx = start_image_idx + num_images
for i in trange(start_image_idx, end_image_idx):
    filepath, image, annotations = ds_train[i]

    labels = [ds_train.classes[class_id] for class_id in annotations.class_id]

    annotated_image = image.copy()
    annotated_image = box_annotator.annotate(annotated_image, annotations)
    annotated_image = label_annotator.annotate(annotated_image, annotations, labels)
    annotated_image = mask_annotator.annotate(annotated_image, annotations)

    print(filepath, annotations)
    sv.plot_image(annotated_image, size=(14, 10))

### Video composition
Generate a video from the exported YOLO-annotated frames.

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
mask_annotator = sv.MaskAnnotator()

start_image_idx = 0
num_images = len(ds_train)
end_image_idx = start_image_idx + num_images
fps = 2
output_path = ROOT_PATH / f"annotated_video-exported-on-{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"

filepath, image, annotations = ds_train[start_image_idx]
h, w = image.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video = cv2.VideoWriter(str(output_path), fourcc, fps, (w, h), isColor=True)

for i in trange(start_image_idx, end_image_idx):
    filepath, image, annotations = ds_train[i]
    labels = [ds_train.classes[class_id] for class_id in annotations.class_id]

    annotated_image = image.copy()
    annotated_image = box_annotator.annotate(annotated_image, annotations)
    annotated_image = label_annotator.annotate(annotated_image, annotations, labels)
    annotated_image = mask_annotator.annotate(annotated_image, annotations)

    video.write(annotated_image)

video.release()
print(f"Video written to '{output_path.resolve()}'")